### Device Check point 

In [20]:
import torch 
import torch.nn as nn
import math 

In [21]:
gpu_available = torch.cuda.is_available()

if gpu_available:
  print(f"GPU is available")

  print(f"GPU count: {torch.cuda.device_count()}")

  print(f"GPU current ID: {torch.cuda.current_device}")

  print(f"GPU name: {torch.cuda.get_device_name(0)}")

else:
  print(f"GPU is not available")

GPU is available
GPU count: 1
GPU current ID: <function current_device at 0x79d2b41f63e0>
GPU name: Tesla T4


In [22]:
### Configuration
class ModelConfig:
    embedding_dim = 768       # size of each token's vector. + We can refer to it as "hidden_size"
    num_heads = 12            # number of attention heads
    head_dim = embedding_dim // num_heads   # size per head = 64
    num_layers = 12           # number of transformer blocks stacked
    vocab_size = 50000        # number of unique tokens the model knows
    max_len = 1024            # max number of tokens in one input
    dropout = 0.1             # regularization rate
    ffn_dim = 3072            # size of the feed-forward layer (usually 4x embedding_dim)
    batch_size = 32           #
    type_vocab_size = 2       # number of segment type like sentence A and Sentence B

cfg = ModelConfig()

print(cfg.type_vocab_size)

2


In [23]:
class BertEmbedding(nn.Module):
  def __init__(self, hidden_size, vocab_size, type_vocab_size, max_len = 512, dropout = 0.2):
    super().__init__()
    self.token_embeddings = nn.Embedding(vocab_size, hidden_size)
    self.position_embeddings = nn.Embedding(max_len, hidden_size)
    self.segment_embeddings = nn.Embedding(type_vocab_size, hidden_size)
    self.layer_norm = nn.LayerNorm(hidden_size, eps = 1e-12)
    self.dropout = nn.Dropout(dropout)

  def forward(self, input_ids, token_type_ids):
    ### input_ids: [batch_size (how many sentence are in this batch), seq_len (how many tokens are in each sentence)]
    seq_len = input_ids.size(1)
    positions = torch.arange(seq_len, device = input_ids.device).unsqueeze(0) ### represent the position indice

    embeddings_summation = self.token_embeddings(input_ids) + self.position_embeddings(positions) + self.segment_embeddings(token_type_ids)

    embeddings_summation = self.layer_norm(embeddings_summation)
    embeddings_summation = self.dropout(embeddings_summation)
    return embeddings_summation


### Scaled-Dot Product and Multi-Head Attention 

In [24]:
class MultiHeadAttention(nn.Module):
  def __init__(self, hidden_size, num_heads, dropout = 0.2):
    super().__init__()
    assert hidden_size % num_heads == 0
    self.num_heads = num_heads
    self.head_dim = hidden_size // num_heads
    # perform y = w^T*x + b to transform identical inputs into distinct specialized role
    self.q_proj = nn.Linear(in_features = hidden_size, out_features = hidden_size, bias = True)
    self.k_proj = nn.Linear(in_features = hidden_size, out_features = hidden_size, bias = True)
    self.v_proj = nn.Linear(in_features = hidden_size, out_features = hidden_size, bias = True)

    self.out_proj = nn.Linear(in_features = hidden_size, out_features = hidden_size, bias = True)
    self.dropout = nn.Dropout(dropout)

  def splits_heads(self, x, batch_size):
    # x: [batch_size, seq_len, hidden_size] --> [batch_size, seq_len, num_heads, head_dim]
    x = x.view(batch_size, -1, self.num_heads, self.head_dim) # reshape the tenspor into [batch_size, wildcard, num_heads, head_dim]
    # x: [0, 1, 2, 3] because each head should process the whole sequence accordingly
    return x.permute(0, 2, 1, 3) # rearrange the tensor to a specific order

  def forward(self, x, attention_mask = None):
    batch_size = x.size(0)

    Q = self.split_heads(self.q_proj(x), batch_size)
    K = self.split_heads(self.k_prof(x), batch_size)
    V = self.split_heads(self.v_proj(x), batch_size)

    scores = torch.matmul(Q, K.tranpose(-1,-2)) # .tranpose(-1, -2) to swap the last two dimensions
    scores = scores / math.sqrt(self.head_dim)

    if attention_mask is not None:
      scores = scores + attention_mask

    attention_weights = torch.softmax(scores, dim = -1)
    attention_weights = self.dropout(attention_weights)
    context = torch.matmul(attention_weights, V)

    context = context.permute(0, 2, 1, 3).contiguous() # permute it back to [0, 1, 2, 3] and physically rearrange the data in the memory
    context = context.view(batch_size, -1, self.num_heads * self.head_dim) # remerging the hidden_size

    return self.out_proj(context) # flow it to be transformed using the nn.Linear

In [ ]:
class FeedForward(nn.Module):
  def __init__(self, hidden_size, intermediate_size, dropout = 0.2):
    super().__init__()
    self.fc1 = nn.Linear(hidden_size, intermediate_size) # expand the vector, giving model more room to transform/combine the feature
    self.fc2 = nn.Linear(intermediate_size, hidden_size) # compress it back
    self.activation = nn.GELU()
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    x = self.activation(self.fc1(x))
    x = self.dropout(x)
    return self.fc2(x)

In [26]:
class TransformerBlock(nn.Module):
  def __init__(self, hidden_size, num_heads, intermediate_size, dropout = 0.2):
    super().__init__()
    self.attention = MultiHeadAttention(hidden_size, num_heads, dropout)
    self.attn_norm = nn.LayerNorm(hidden_size, eps = 1e-12)

    self.ffn = FeedForward(hidden_size, intermediate_size, dropout)
    self.ffn_norm = nn.LayerNorm(hidden_size, eps = 1e-12)

    self.dropout = nn.Dropout(dropout)

  def forward(self, x, attention_mask = None):
    attn_out = self.attention(x, attention_mask)
    x = self.attn_norm(x + self.dropout(attn_out)) # residual connection

    ffn_out = self.ffn(x)
    x = self.ffn_norm(x + self.dropout(ffn_out)) # residual connection
    
    return x

In [28]:
class BertFromScratch(nn.Module):
  def __init__(self, vocab_size, hidden_size, num_heads =12, num_layers = 12, intermediate_size = 3072, max_len = 512, dropout = 0.2):
    super().__init__()
    self.embeddings = BertEmbedding(hidden_size, vocab_size, type_vocab_size, max_len,  dropout) 
    self.layers = nn.ModuleList([
                TransformerBlock(hidden_size, num_heads, intermediate_size, dropout)
                for _ in range(num_layers)
            ])
    self.pooler = nn.Linear(hidden_size, hidden_size) # for the [CLS] pooling
    self.pooler_act = nn.Tanh()

  def forward(self, input_ids, token_type_ids = None, attention_mask = None):
    if token_type_ids is None:
      token_type_ids = torch.zeros_likes(input_ids) # fill entirely with zero but keep everything like the datatype, shape, layout, device the same

    if attention_mask is not None:
      ext_mask = attention_mask[:, None, None, :].float()
      ext_mask = (1 - ext_mask) * (-10000)
    else:
      ext_mask = None

    x = self.embeddings(input_ids, token_type_ids)
    for layer in self.layers:
      x = layer(x, ext_mask)

    sequence_output = x 
    cls_token = sequence_output[:, 0]
    pooled_output = self.pooler_act(self.pooler(cls_token))

    return sequence_output, pooled_output

In [29]:
class BertLMPredictioonHead(nn.Module):
    def __init__(self, hidden_size, vocab_size, token_embedddings: nn.Embedding):
        super().__init__()
        self.dense = nn.Linear(hidden_size, hidden_size)
        self.activation = nn.GELU()
        self.layer_norm = nn.LayerNorm(hidden_size, eps =1e-12)

        self.decoder = nn.Linear(hidden_size, vocab_size, bias = False)
        self.decoder.weight = token_embedddings.weight
        self.bias = nn.Parameter(torch.zero(vocab_size))

    def forward(self, sequence_output):
        x = self.activation(self.dense(sequence_output))
        x = self.layer_norm(x)
        logits = self.decoder(x) + self.bias
        return logits

In [30]:
class BertNSPHead(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, 2)

    def forward(self, pooled_output):
        return self.classifier(pooled_output) # [batch_size, 2]

In [31]:
class BertForPreTraining(nn.Module):
    def __init__(self, bert: BertFromScratch, vocab_size, hidden_size = 768):
        super().__init__()
        self.bert = bert 
        self.mlm_head = BertLMPredictioonHead(hidden_size, vocab_size, self.bert.embeddings.token_embeddings)
        self.nsp_head = BertNSPHead(hidden_size, 2)

    def forward(self, input_ids, token_type_ids = None, attention_mask = None, mlm_labels = None, nsp_labels = None):
        sequence_output, pooled_output = self.bert(input_ids, token_type_ids, attention_mask)

        mlm_logits = self.mlm_head(sequence_output)
        nsp_logits = self.nsp_head(pooled_output)

        loss = None 

        if mlm_labels is not None and self.nsp_labels is not None:
            # mlm_labels should be -100 at every position except the masked
            # tokens you actually want predicted -- cross_entropy skips
            # those automatically via ignore_index.
            mlm_loss = F.cross_entropy(
                mlm_logits.view(-1, mlm_logits.size(-1)),
                mlm_labels.view(-1),
                ignore_index = -100,
            )
            nsp_loss = F.cross_entropy(nsp_logits, nsp_labels)

            loss = mlm_loss + nsp_loss


        return {"mlm_logits": mlm_logits, "nsp_logits": nsp_logits, "loss": loss}


In [32]:
class BertForMaskedLM(nn.Module):
    def __init__(self, bert: BertFromScratch, vocab_size, hidden_size=768):
        super().__init__()
        self.bert = bert
        self.mlm_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, input_ids, token_type_ids=None, attention_mask=None):
        sequence_output, _ = self.bert(input_ids, token_type_ids, attention_mask)
        logits = self.mlm_head(sequence_output)  # [batch, seq, vocab_size]
        return logits

In [ ]:
if __name__ == "__main__":
    torch.manual_seed(0)

    # Small config so this runs fast on CPU -- swap in your ModelConfig
    # (embedding_dim=768, num_layers=12, vocab_size=50000, ...) for the real thing.
    cfg = ModelConfig()
    cfg.embedding_dim = 128
    cfg.num_heads = 4
    cfg.num_layers = 2
    cfg.vocab_size = 3000
    cfg.max_len = 64

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    bert = BertFromScratch(
        vocab_size=cfg.vocab_size,
        hidden_size=cfg.embedding_dim,
        num_heads=cfg.num_heads,
        num_layers=cfg.num_layers,
        intermediate_size=cfg.ffn_dim,
        max_len=cfg.max_len,
        type_vocab_size=cfg.type_vocab_size,
        dropout=cfg.dropout,
    ).to(device)

    model = BertForPreTraining(bert, vocab_size=cfg.vocab_size, hidden_size=cfg.embedding_dim).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model created with {n_params:,} parameters")
    print(f"Config: hidden={cfg.embedding_dim}, layers={cfg.num_layers}, heads={cfg.num_heads}\n")

    # ---- Fake a mini pretraining batch ----
    batch_size, seq_len = 4, 16
    input_ids = torch.randint(0, cfg.vocab_size, (batch_size, seq_len), device=device)
    token_type_ids = torch.cat(
        [torch.zeros(batch_size, seq_len // 2), torch.ones(batch_size, seq_len // 2)], dim=1
    ).long().to(device)
    attention_mask = torch.ones(batch_size, seq_len, device=device)
    attention_mask[:, -3:] = 0  # pretend the last 3 tokens are padding

    mlm_labels = torch.full((batch_size, seq_len), -100, device=device)
    mlm_labels[:, 3] = torch.randint(0, cfg.vocab_size, (batch_size,), device=device)  # masked position 3
    mlm_labels[:, 7] = torch.randint(0, cfg.vocab_size, (batch_size,), device=device)  # masked position 7

    nsp_labels = torch.randint(0, 2, (batch_size,), device=device)

    out = model(
        input_ids=input_ids,
        token_type_ids=token_type_ids,
        attention_mask=attention_mask,
        mlm_labels=mlm_labels,
        nsp_labels=nsp_labels,
    )

    print("MLM logits shape:", out["mlm_logits"].shape)  # [batch_size, seq_len, vocab_size]
    print("NSP logits shape:", out["nsp_logits"].shape)  # [batch_size, 2]
    print("Pretraining loss:", out["loss"].item())

    out["loss"].backward()
    grad_norm = sum(p.grad.norm() for p in model.parameters() if p.grad is not None)
    print(f"Total gradient norm after backward(): {grad_norm:.4f}")

### BERT using PyTorch 

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

text = "I am tinkering around with BERT!"

encoded = tokenizer(text, padding = "max_length", truncation = True, return_tensors = "pt", max_length =16)

print(encoded["input_ids"])
print(encoded["attention_mask"])

# 101 = [CLS] 
# 102 = [SEP]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


tensor([[  101,  1045,  2572,  9543,  5484,  2075,  2105,  2007, 14324,   999,
           102,     0,     0,     0,     0,     0]])
tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0]])


In [11]:
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-uncased")

outputs = model(**encoded)

last_hidden_state = outputs.last_hidden_state
cls_embedding = last_hidden_state[:, 0 , :]


print(last_hidden_state.shape) # [batch_size, seq_len, hidden_size]



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 16, 768])


In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_dataset
import warnings
warnings.filterwarnings("ignore")

from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

dataset = load_dataset("stanfordnlp/imdb")

def tokenize_fn(batch):
  return tokenizer(batch["text"], padding = "max_length", truncation = True, max_length = 128)

dataset = dataset.map(tokenize_fn, batched = True)

dataset.set_format(type = "torch", columns = ["input_ids", "attention_mask", "label"])

model = BertForSequenceClassification.from_pretrained("bert-base-uncased")

training_arg = TrainingArguments(
    output_dir = "./results",
    per_device_train_batch_size = 8,
    num_train_epochs = 2,
    eval_strategy = "epoch",
    logging_steps = 50,
)


trainer = Trainer(
    model = model,
    args = training_arg,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
)

trainer.train()